# Gates

Every check the reporting code is held to, run one at a time with its output in view.

**The `.py` files under `test/` remain the source of truth** — this notebook runs them,
it does not restate them. A gate lives in exactly one place, so an assertion cannot pass
here and fail there. What the notebook adds is visibility: each file's printed verdict
beside a short statement of what it is for, and, at the end, the objects the gates assert
about rendered as tables and figures so you can see the fixture rather than infer it.

None of it needs a run tree, Julia, or the real data. Every fixture is planted here, with
answers chosen so that they need no arithmetic to check.

Run the whole notebook, and send me the output when something goes red.


In [1]:
import io, os, subprocess, sys, time, traceback
from contextlib import redirect_stdout, redirect_stderr
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "utils.py").exists() and (ROOT.parent / "utils.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "test"))

RESULTS = {}


def run_gate(name, show="all"):
    """Run one gate file in a fresh interpreter and show what it printed.

    A SUBPROCESS, not an import: the gate files install fixture stubs onto the library
    modules (that is how a planted economy replaces `sourcing_geometry`), so running two
    of them in one interpreter would let one file's stubs leak into the next. A fresh
    process per gate is the only way to keep them independent, and it is what running
    them from the shell does anyway.

    `show="all"` prints everything, `show="tail"` the last lines, `show="verdict"` the
    ok/FAIL lines alone.
    """
    t0 = time.time()
    p = subprocess.run([sys.executable, str(ROOT / "test" / name)],
                       capture_output=True, text=True, cwd=str(ROOT / "test"))
    out = (p.stdout or "") + (p.stderr or "")
    ok = p.returncode == 0
    RESULTS[name] = {"ok": ok, "seconds": time.time() - t0,
                     "gates": sum(1 for L in out.splitlines()
                                  if (L.strip()[:1].isdigit() and " ok" in L[:40])
                                  or L.strip().startswith("PASS")),
                     "output": out}
    lines = out.splitlines()
    if show == "verdict":
        lines = [L for L in lines if " ok " in L or "ok " == L[:3] or "FAIL" in L
                 or "Error" in L]
    elif show == "tail":
        head, lines = lines[:2], lines[-25:]
        n_elided = len(out.splitlines()) - len(head) - len(lines)
        lines = head + [f"... ({n_elided} lines elided; re-run with show='all') ..."] + lines
    display(Markdown(f"### {'PASS' if ok else 'FAIL'} — `{name}`  "
                     f"({RESULTS[name]['gates']} gates, "
                     f"{RESULTS[name]['seconds']:.1f}s)"))
    print("\n".join(lines) if lines else "(no output)")
    if not ok:
        display(Markdown("**This one is red. Send me the block above.**"))
    return ok


def summary():
    df = pd.DataFrame(RESULTS).T[["ok", "gates", "seconds"]]
    df["seconds"] = df["seconds"].astype(float).round(1)
    return df


## The module layout, the loader and the parquet-free path

Three things no section gate can check on its own.

1. **One definition per name** across `utils.py`, `report_lib.py`, `diffusion_lib.py` and
   `granular_lib.py`. The notebooks used to carry 75 duplicated definitions and the gates
   policed the copies for drift; this makes the duplication impossible instead.
2. **The specialised loader.** A run tree with the moment, inference, Jacobian and count
   artefacts *deleted* must open under `parts=("core", "geography")` — and asking for a
   part whose files are missing must fail, so the guard is not vacuous.
3. **The parquet-free path**, and that the counterfactual lets the **spending shares
   respond**: the factorisation `X_lr = (D_r − 1) · Σ_s θ_rs ρ_lrs` is exact, `D_r` moves
   with the regime, and the fixed-spend route is the geography channel alone.

In [2]:
run_gate("test_modules.py")


### PASS — `test_modules.py`  (5 gates, 4.0s)

1 ok  227 functions across 4 modules, none defined twice
2 ok  a tree with no moment, inference or Jacobian artefact opens under parts=('core','geography'); asking for a part whose files are absent fails, and 'jacobian' pulls in what it is read against
3 ok  every regime is built from theta+ with no parquet in the tree; D_r closes against the value block (1.7594) and now MOVES across regimes (range 0.2572) where the closed-form reallocation holds it fixed, the local share moves under both, and the continuum benchmark is simulated (-0.4 origins of granularity)
4 ok  the counterfactual lets the spending shares respond: the factorisation X = (D_r-1) * sum_s theta_rs rho_lrs is exact to 1.7e-16, D_r moves with the regime (range 0.2602) where the fixed-spend route pins it, and the sector MIX moves the local share (5.20e-04) beside the geography (8.27e-02)
5 ok  the local-share figure accepts any radii (the default sort key derives from them), and `reporting_data` resolves n_rep before its c

True

## The two notebooks

Statically, every code cell parses and every free name it uses is bound — by the libraries
it star-imports or by an earlier cell. That is the property the old single-namespace
layout gave for free and that a notebook of imports has to earn: a function left behind in
the other notebook would surface as a `NameError` only when its cell was run.

Functionally, the imports, the Constants cell and the economy run cell execute against a
run tree carrying no `suppliers.parquet`.

In [3]:
run_gate("test_notebooks.py")


### PASS — `test_notebooks.py`  (4 gates, 1.8s)

1 ok  2 notebooks, 28 code cells: every cell parses and every free name is bound by the libraries or an earlier cell
2 ok  both notebooks specialise the loader, neither asks for the `firm` part, so suppliers.parquet is never opened on a reporting path
2b ok  Test 9 is merged into Test 6: 16 test headings, both halves in Test 6's markdown, both halves called by its run cell, and the header agrees

=== [test, mu = 2] ==============================================
  [theta+] Omega_L 0.3100  alpha 0.4000  theta 1.7680 (load_parameters.jl)  N_s 4-7  eps -16.000  nu 0.2/1.5  lam 0.5
  [julia] no post_hoc_u.npy / suppliers.parquet in this tree, so the port is unverified against solve_network. Run main.jl's post-hoc block once to get that check; it is not needed for anything below.
  Both forces                  D_r 1.7594  P_r 2.0765  n_rep 3  draws rng(20260912)
  Distance only                D_r 1.7563  P_r 2.0330  n_rep 3  draws rng(20260912)
  Comparative advantage only   D_r 1.5021  P_r 

True

## The extended parameter set, and the economy it defines

`θ⁺ = (Ω_L, Ω_s, A, α, T, N)` plus the fixed calibration, and `simulate_economy` is the
forward map from it to a realised finite-variety economy. Because the counterfactuals are
produced by the *same* code as the estimated economy, what has to be gated is that the map
**closes** (the two CES identities, the input mix, `D_r`) against arithmetic recomputed
independently, that it is a **deterministic** function of `θ⁺` and the draws, that
averaging it returns the closed-form `ρ`, and that the two exact controls hold — `α = 0`
makes every buyer pick the same winner, and `N` does not move across regimes.

Gate 9 is the cross-language comparator against a parquet written in Julia's schema; gate
11 is the round trip through the notebook's own reader.

In [4]:
run_gate("test_extended_economy.py")


### PASS — `test_extended_economy.py`  (12 gates, 2.1s)

1 ok  the good index runs region-outer/sector-inner as Julia's column-major findall does (24 cells), which is what aligns the draw columns
2 ok  the two CES identities, the input mix and D_r all close to machine precision (worst 6.7e-16), and the panel carries sector keys only
3 ok  P_r, c_r, c_tilde_r, D_r, theta_rs, P and Y_r reproduce a hand recomputation from theta+ (D_r median 1.637) -- and D_r IS 1 + (1-Omega_L)(P_r/c_r)^(1-lambda), the closed form
4 ok  the economy is a deterministic function of (theta+, draws); the head moves the value block and leaves the winners untouched, which is why the Ricardian half needs only (alpha, T, N)
5 ok  the realised winner frequencies converge to the closed-form rho: worst |z| 3.29 over 120 cells at 400 replications
6 ok  alpha=0 gives one winner per variety for every buyer (the exact control), equalising T sends the euro further (218 -> 227 km), and N is held at theta+'s value in every regime
7 ok  a supplied (N_max, n_good) draw matrix reprod

True

## Concentration and commonality

A planted economy whose answers need no arithmetic — equal variety expenditure, so
`V = 1/N_s` **exactly** — plus every identity the code claims: the decomposition
`H̄ = H(ω̄) + M`, the `α = 0` control giving `C = 1` to machine precision, the uniform
benchmark returning `n_eff = n_cells`, the four cells adding up, and `ρ` hitting its floor
when winners are independent.

In [5]:
run_gate("test_concentration_identity.py")


### PASS — `test_concentration_identity.py`  (21 gates, 9.3s)

1 ok  identity, alpha=0 gives C=1 exactly, uniform gives n_eff = n_cells
2 ok  the buyer aggregation reproduces the industry number
3 ok  derivative identity residual 1.26e-08
4 ok  four cells add up; E[H] formula within 1.671%; granular addition [0.176 0.081 0.029] falls with N_s
5 ok  E[omega] vs rho: median TV 0.0331
6 ok  the zone attribution of M sums to one
7 ok  table and both figures render; rows keep the table order by default
8 ok  concentration_report wires every piece together
9 ok  V = 1/N_s exactly under equal variety shares, buyer-independent
10 ok unequal shares give V x N = [1.29, 1.32, 1.34] > 1 (Cauchy-Schwarz)
11 ok independent winners give Q = G; shared winners give Q = 1
12b ok equal weights put buyer_hhi on 1/n exactly; the size weighting reproduces emp_pi_r on the support and is strictly more concentrated (1/hhi 3.62 vs 5.00)
12 ok four cells add up; rho hits its floor 0.20 when winners are independent and >0.9 when shared; the variety route reproduces the reali

True

## The local share: a level against a granular dispersion

The central gate is that two routes sharing no code agree: the measured standard deviation
across replications, and the closed form `√(Σ_s θ² V p(1−p))`. Plus the shape result —
the dispersion peaks where `p` is near one half — gated on the **draws**, where it is a
finding, rather than on the closed form, where it would be a tautology.

In [6]:
run_gate("test_local_share_dispersion.py")


### PASS — `test_local_share_dispersion.py`  (10 gates, 4.2s)

1 ok  the own-zone indicator picks exactly the buyer's zone (8 buyers have one in that sector, 7 do not) and the level is sum_s theta gamma
2 ok  V = 1/N_s exactly on the planted panel, and the table's level and V_r are the theta-weighted sector sums, with p x N_eff = local / V_r
3 ok  measured sd vs the SQUARED-weight closed form: median ratio 1.007, max deviation 0.034 at B = 200; max |z| on the level 0.53
4 ok  the band vanishes exactly at p = 0 and p = 1 -- width and measured sd both -- and is strictly positive in between
5 ok  the MEASURED dispersion tracks sqrt(V p(1-p)) to a median 2.2% and peaks in the bin containing p = 1/2, falling to 64% and 51% of its peak at the two ends
6 ok  coverage is bounded below by 0.80 and above by 1 (median 0.800), equals the share of draws inside the drawn arms, and rises to 0.840 on a coarser lattice
  [local share] Both forces: sectors [1] carry no varieties — the AGGREGATE measured dispersion is dropped for this regime; the per-sector rows tha

True

## The alignment covariance, in kilometres

`Cov_ρ(log T, d)` — the rate the `Distance only` counterfactual integrates. Gate 1 is the
regression this file exists for, restated: `alignment_frame` and `_buyer_weights` used to
exist in two byte-identical copies and the gate checked they had not drifted; now it
checks there **is** only one of each.

In [7]:
run_gate("test_alignment_covariance.py")


### PASS — `test_alignment_covariance.py`  (6 gates, 2.1s)

1 ok  one definition each, across 4 modules and 227 functions -- the two copies this gate used to police are gone
2 ok  the panel is 72 = (cells x buyers) rows, rho sums to one per (sector, buyer), weight to one over the panel, and the distances are the geometry's own
3 ok  the covariance reproduces a raw-moment recomputation to 1e-9, Cov(log d, d) > 0 everywhere, and the two weightings differ
4 ok  the planted aligned sector has a negative median (-35 km, 75% of buyers negative) against +0 km for the orthogonal one — the sign the counterfactual integrates, and the per-buyer flip the aggregate hides
5 ok  the buyer aggregation is the spend-weighted mean of its sectors, differs from a plain mean under lopsided spending, and refuses both a missing parquet and a non-matching region index
6 ok  one density and one median rule per industry on one panel, an honest zero, the buyer level, and a refusal for an unknown level or weighting

all gates pass


True

## The supplier's portfolio of customers

The mirror object: tests 1–7 read a buyer's portfolio of suppliers, this reads a
supplier's portfolio of customers. The fixture plants two segmented blocks so the
partition has a known answer, and the rewiring null has something to fail against — it
degenerates on 98% of draws, which is reported rather than hidden.

In [8]:
run_gate("test_portfolio_similarity.py")


### PASS — `test_portfolio_similarity.py`  (6 gates, 2.4s)

1 ok  cosine is symmetric, in [-1,1], invariant to portfolio scale and to the share normalisation, but NOT to the buyer weighting; empty portfolios are dropped
2 ok  the planted blocks are recovered at tau = 0.8; the group count rises with tau [np.int64(1), np.int64(1), np.int64(3), np.int64(6)] and collapses to one component at tau = 0 (the chaining the text names)
3 ok  ordered-pair denominators reproduce a hand recomputation; singletons are absent from the within average and present in the between one (within 0.987, between 0.309, delta 0.679)
4 ok  rewiring preserves every degree; the permutation null is finite on every draw (mean -0.021) and the planted delta +0.679 clears it, while the rewiring null degenerates on 98% of draws
5 ok  the segmented sector separates at tau >= 0.7 (excess +0.673) and chains into one group at 0.6; the common-clientele sector admits no partition at any tau despite the HIGHER mean similarity (1.000 against 0.525), and the rewiring null is flagged degene

True

## The moment layout and both Jacobian axes

The largest file, and the one that needs no data: it writes a synthetic baseline +
reporting tree with the exact file layout `main.jl --granular=true --ca_level=aa`
produces, then runs the shipped modules against it. Two industries, because the joint
table's `---` path is only exercised when their sector sets differ.

In [9]:
run_gate("test_analysis_granular.py", show="tail")


### PASS — `test_analysis_granular.py`  (169 gates, 8.7s)

synthetic test: /tmp/tmpc744u1bx  (S=4, R=12, n_AA=3, n_gb=15, gamma kept=7)
synthetic test2: /tmp/tmpc744u1bx  (S=4, R=12, n_AA=3, n_gb=15, gamma kept=7)
... (194 lines elided; re-run with show='all') ...
  PASS  elasticity Jacobian is the one plotted
  PASS  block summary is moment-blocks x (statistic, parameter-block)
  PASS  noise-to-signal ratio is sigma/|elasticity|, exact zeros counted as measured
  PASS  nothing is masked at the default threshold, everything at a strict one
  PASS  triptych writes the matrix, the noise map and the purged matrix
  PASS  plot_jacobian_noise -> pdf
  PASS  plot_identification_map -> pdf
  PASS  plot_channel_elasticities -> pdf
  PASS  plot_jacobian_thresholded -> pdf
  PASS  identification summary separates structural zeros from weak channels
  PASS  plot_jacobian_full -> pdf
  PASS  plot_jacobian_blocks -> pdf

=== 13. variance-covariance ===
  PASS  Sigma_data / Sigma_sim / Omega / W_step3 loaded
  PASS  all are n_gb x n_gb
  PASS  summary cover

True

## The reporting sections

Identification, the untargeted moment, the input-output benchmark, comparative advantage
and amplification, each on its own planted fixture.

In [10]:
run_gate("test_analysis_granular_sections.py", show="tail")


### PASS — `test_analysis_granular_sections.py`  (0 gates, 13.0s)


... (265 lines elided; re-run with show='all') ...
ALL GATES PASSED
/usr/local/lib/python3.11/dist-packages/pyfixest/estimation/models/feols_.py:2513: UserWarning: 
            1 variables dropped due to multicollinearity.
            The following variables are dropped: ['log_productivity'].
            
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 722 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/home/user/SMM_Spatial_Comovement/diffusion_lib.py:1145: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=figsize or get_figsize(wf=0.8, hf=0.6))
/home/user/SMM_Spatial_Comovement/diffusion_lib.py:1613: Us

True

## The identities of `documentation/diversification.md`

Standalone — numpy only, no library. Every displayed identity of that note re-derived by
finite differences, Monte Carlo or exact arithmetic, on asymmetric random economies so a
transposed index shows up.

In [11]:
run_gate("test_diversification_identities.py", show="tail")


### PASS — `test_diversification_identities.py`  (28 gates, 0.4s)

Section 3 -- the Herfindahl representation
... (23 lines elided; re-run with show='all') ...
PASS  P8  the derivatives sum to zero over buyers
PASS  S5  at alpha = 0 the market portfolio is the observed expenditure share

Section 4.3 -- the extensive margin (Monte Carlo)
PASS  P9  marginal win probability equals gamma
PASS  P9  eq (14) E[n] = sum_r gamma
PASS  P9  eq (16) unconditional FKG bound holds for every origin
PASS  P9  eq (13) E_u[exp(-kappa u)] = gamma (champion convention)

Sections 5 and 6
PASS  P10 eq (18) tilting identity
PASS  A.11 Htilde = H + Cov_p(nu, a) / E_p[nu]
      motor vehicles, data     delta/gamma=-0.1090  TF=0.404  H=0.596  N=1.68
      aerospace, data          delta/gamma=-0.0980  TF=0.228  H=0.772  N=1.30
      motor vehicles, model    delta/gamma=-0.0846  TF=0.313  H=0.687  N=1.46
      aerospace, model         delta/gamma=-0.0461  TF=0.107  H=0.893  N=1.12
PASS  S5  table reproduces (conversion at mean log d = 5.8)
PASS  P11 eq (20) exact when the upstre

True

---

## Summary

One row per gate file. A red row is what to send me — the block above it carries the
traceback and the last assertion that held, which is usually enough to locate the
failure without running anything.

In [12]:
# Everything at a glance. Red rows are what to send me.
display(summary())
n_bad = int((~summary()["ok"].astype(bool)).sum())
print(f"\n{len(RESULTS)} gate files, {int(summary()['gates'].sum())} numbered gates, "
      f"{n_bad} failing")


,ok,gates,seconds
test_modules.py,True,5,4.0
test_notebooks.py,True,4,1.8
test_extended_economy.py,True,12,2.1
test_concentration_identity.py,True,21,9.3
test_local_share_dispersion.py,True,10,4.2
test_alignment_covariance.py,True,6,2.1
test_portfolio_similarity.py,True,6,2.4
test_analysis_granular.py,True,169,8.7
test_analysis_granular_sections.py,True,0,13.0
test_diversification_identities.py,True,28,0.4



10 gate files, 261 numbered gates, 0 failing


---

# What the gates assert about

The rest of the notebook is not a check — it renders the objects the gates reason over, so
that a number in an assertion can be read against the thing it constrains.

In [13]:
# --- What the gates are asserting ABOUT -----------------------------------------------
# The planted economy of `test_extended_economy.py`: 3 sectors, 12 regions, 5 buyers, and
# a hand-written `theta+`. Nothing here is read from a run tree -- the point of writing
# `theta+` down is that it is a plain object.
import warnings; warnings.filterwarnings("ignore")
matplotlib.use("Agg")
sys.path.insert(0, str(ROOT / "test"))
import utils, granular_lib, diffusion_lib
from _nbmod import install

S, R, THETA, ALPHA = 3, 12, 1.3, 0.4
rng = np.random.default_rng(3)
CELL_MASK = np.zeros((S, R), bool)
for s in range(S):
    CELL_MASK[s, rng.choice(R, 8, replace=False)] = True
buyers = np.array([1, 2, 3, 4, 5])
D = rng.uniform(20, 600, (R, R)); np.fill_diagonal(D, 10.)
D = (D + D.T) / 2
Tcell = {s: rng.lognormal(0, .6, size=int(CELL_MASK[s].sum())) for s in range(S)}
N_HAT = np.array([4, 10, 30])


def sourcing_geometry(data, alpha=None, equalise_T=False):
    a = ALPHA if alpha is None else float(alpha)
    out = {}
    for s in range(S):
        cells = np.flatnonzero(CELL_MASK[s])
        T = np.ones(cells.size) if equalise_T else Tcell[s]
        d = np.maximum(D[np.ix_(cells, buyers - 1)], 1.0)
        psi = T[:, None] * d ** (-THETA * a)
        out[s] = {"cells": cells, "T_cell": T, "distance": d,
                  "rho": psi / psi.sum(0, keepdims=True)}
    return {"by_sector": out, "alpha": a, "theta": THETA, "downstream": buyers}


NU_S_DEFAULT, NU_ACROSS_DEFAULT, LAMBDA_DEFAULT = 1.5, 0.2, 0.5
data = {"S": S, "R": R, "CELL_MASK": CELL_MASK, "post_hoc_N_hat": N_HAT,
        "folder": "x", "step_dir": "step3"}
install(globals(), [utils, granular_lib, diffusion_lib])

XP = {"Omega_L": 0.31, "Omega_s": np.array([0.2, 0.3, 0.5]),
      "A": np.array([1.0, 1.4, 0.8, 1.1, 0.9]), "alpha": np.array([ALPHA]),
      "T": None, "N": N_HAT.copy(), "theta": THETA, "theta_source": "fixture",
      "nu_s": np.full(S, NU_S_DEFAULT), "nu": NU_ACROSS_DEFAULT, "lam": LAMBDA_DEFAULT,
      "epsilon": -16.0, "delta": np.ones(buyers.size), "wage": np.ones(R)}
print("theta+ :", {k: (v if np.isscalar(v) else np.shape(v)) for k, v in XP.items()})


theta+ : {'Omega_L': 0.31, 'Omega_s': (3,), 'A': (5,), 'alpha': (1,), 'T': (), 'N': (3,), 'theta': 1.3, 'theta_source': 'fixture', 'nu_s': (3,), 'nu': 0.2, 'lam': 0.5, 'epsilon': -16.0, 'delta': (5,), 'wage': (12,)}


## The forward map, and the identities it closes

In [14]:
# The economy the map produces, and the identities it must satisfy EXACTLY. Each one
# fixes a different step of the forward map, so a failure names the step.
econ = simulate_economy(data, XP, n_rep=20, seed=11)
res = economy_identities(econ, XP, verbose=True)
display(pd.Series(res, name="worst absolute error").to_frame().style.format("{:.2e}"))
print("\nvalue block, averaged over draws:")
display(pd.DataFrame({k: econ.value[k].mean(axis=0) for k in
                      ("P_r", "c_r", "c_tilde_r", "D_r", "Y_r")},
                     index=[f"buyer {b}" for b in econ.meta["buyers"]]).round(4))


  [identities] ces_within 5.55e-16  ces_across 2.22e-16  mix_sums_one 2.22e-16  D_r_from_shares 4.44e-16  -> ok


,worst absolute error
ces_within,5.55e-16
ces_across,2.22e-16
mix_sums_one,2.22e-16
D_r_from_shares,4.44e-16



value block, averaged over draws:


,P_r,c_r,c_tilde_r,D_r,Y_r
buyer 1,0.6482,0.7480,0.7480,1.6402,0.0209
buyer 2,0.7310,0.8087,0.5776,1.6539,0.9075
buyer 3,0.6330,0.7366,0.9207,1.6372,0.0009
buyer 4,0.6995,0.7854,0.7140,1.6483,0.0576
buyer 5,0.5952,0.7087,0.7875,1.6307,0.0131


## The three regimes, and the channel decomposition

In [15]:
# The three regimes off one `theta+`, and the channel decomposition the amplification
# section reads. D_r moves through the LEVEL channel alone; the local share and the mean
# distance are ratios in which (D_r - 1) cancels, so they move through the sector MIX and
# the within-sector GEOGRAPHY alone.
ECON3 = {}
for lab, kw in CF_REGIMES.items():
    e = simulate_economy(data, XP, n_rep=20, seed=11, **kw)
    ECON3[lab] = {**simulated_data(data, e, XP), "economy": e}

rows = []
for lab, dl in ECON3.items():
    V = dl["economy"].value
    rows.append({"regime": lab, "D_r": V["D_r"].mean(), "P_r": V["P_r"].mean(),
                 "c_tilde_r": V["c_tilde_r"].mean()})
display(pd.DataFrame(rows).set_index("regime").round(4))


,D_r,P_r,c_tilde_r
regime,,,
Both forces,1.6420,0.6614,0.7496
Distance only,1.6825,0.9484,0.9533
Comparative advantage only,1.3893,0.0829,0.2576


In [16]:
# `_region_labels` reads a run tree, so the decomposition is exercised here on its own
# arithmetic rather than through the reporting wrapper.
mix = {lab: dl["economy"].value["theta_rs"].mean(axis=0) for lab, dl in ECON3.items()}
geo = {lab: sourcing_geometry(data, **CF_REGIMES[lab]) for lab in ECON3}
base = "Both forces"
rows = []
for lab in ECON3:
    tot = _regime_profile(data, geo[lab], mix[lab], (200,))
    gch = _regime_profile(data, geo[lab], mix[base], (200,))     # mix held at baseline
    mch = _regime_profile(data, geo[base], mix[lab], (200,))     # geometry held
    b = _regime_profile(data, geo[base], mix[base], (200,))
    rows.append({"regime": lab,
                 "D_r": ECON3[lab]["economy"].value["D_r"].mean(),
                 "d_r": tot["mean_upstream_distance"].mean(),
                 "L_r(200) total": tot["share_within_200km"].mean(),
                 "  of which geography": (gch - b)["share_within_200km"].mean(),
                 "  of which mix": (mch - b)["share_within_200km"].mean()})
display(pd.DataFrame(rows).set_index("regime").round(4))
print("The last two columns are DEVIATIONS from the baseline; the first three are levels.\n"
      "D_r moving across regimes is the correction: equalising comparative advantage\n"
      "changes how much of the euro is bought upstream at all, not only where it goes.")


,D_r,d_r,L_r(200) total,of which geography,of which mix
regime,,,,,
Both forces,1.6420,244.9701,0.2871,0.0000,0.0000
Distance only,1.6825,251.0678,0.3017,0.0077,0.0104
Comparative advantage only,1.3893,311.1403,0.1510,-0.1550,0.0471


The last two columns are DEVIATIONS from the baseline; the first three are levels.
D_r moving across regimes is the correction: equalising comparative advantage
changes how much of the euro is bought upstream at all, not only where it goes.


## A figure

In [17]:
# The figures the gates draw, shown rather than counted. What the gates assert about them
# is structural -- one series per regime, an honest zero, the bar equal to k sigma in data
# units -- and is checked in `test_local_share_dispersion.py`; this is the picture.
matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    disp = local_share_dispersion(data, n_rep=60, seed=7, verbose=False)
    axes = plot_local_share_dispersion(disp)
    plt.show()
except Exception as e:
    print(f"(skipped: {type(e).__name__}: {e})")


(skipped: TypeError: local_share_dispersion() got an unexpected keyword argument 'n_rep')
